In [1]:
%run utils.ipynb

In [2]:
import requests
import logging
import os
import json
from datetime import datetime
import time

# --- FUNÇÕES AUXILIARES PARA NOTEBOOK ---
def now_timestamp() -> str:
    """Gera um timestamp formatado."""
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def write_json(data: list, path: str):
    """Salva a lista de dicionários em um arquivo JSON."""
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
# ------------------------------------------

# URL da API v2 do Open Brewery DB
API_URL = 'https://api.openbrewerydb.org/v1/breweries'
PER_PAGE = 50  # Máximo permitido por página

# Configura o logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
BASE_PATH = './data_lake' # Definição do path para o ambiente local/Docker

def fetch_breweries():
    """Busca todas as cervejarias da API usando paginação e retorna uma lista de dicionários."""
    all_items, page = [], 1
    
    while True:
        try:
            r = requests.get(API_URL, params={'page': page, 'per_page': PER_PAGE}, timeout=30)
            r.raise_for_status()
            items = r.json()

            if not items:
                logging.info(f'No more data at page {page}. Ending fetch.')
                break

            all_items.extend(items)
            logging.info(f'Fetched page {page} with {len(items)} records')

            # Verifica se é a última página (menos itens que o limite)
            if len(items) < PER_PAGE: 
                logging.info('Last page reached.')
                break
                
            page += 1
            time.sleep(0.5) # Boa prática para evitar sobrecarregar a API

        except requests.RequestException as e:
            logging.error(f'Failed to fetch page {page} - API Error: {e}')
            # No Airflow/Orchestrator, esta seria uma falha com retry.
            break
        except Exception as e:
            logging.exception(f'An unexpected error occurred: {e}')
            break

    logging.info(f'Total records fetched: {len(all_items)}')
    return all_items


def run_extract(output_dir=BASE_PATH):
    """Executa o processo de extração para a Bronze Layer."""
    out_dir = os.path.join(output_dir, 'bronze')
    os.makedirs(out_dir, exist_ok=True)

    # Cria o nome do arquivo com o timestamp
    out_path = os.path.join(out_dir, f'breweries_{now_timestamp()}.json')

    data = fetch_breweries()
    
    if data:
        write_json(data, out_path)
        logging.info(f'Wrote {len(data)} records to {out_path}')
        return out_path
    
    return None

# ---- EXECUÇÃO (Célula 1) ----
bronze_file_path = run_extract()
print(f"\nExtração concluída! Arquivo salvo em: {bronze_file_path}")

2025-10-21 12:20:00,981 - INFO - Fetched page 1 with 50 records
2025-10-21 12:20:01,790 - INFO - Fetched page 2 with 50 records
2025-10-21 12:20:03,271 - INFO - Fetched page 3 with 50 records
2025-10-21 12:20:09,090 - INFO - Fetched page 4 with 50 records
2025-10-21 12:20:10,798 - INFO - Fetched page 5 with 50 records
2025-10-21 12:20:13,081 - INFO - Fetched page 6 with 50 records
2025-10-21 12:20:15,284 - INFO - Fetched page 7 with 50 records
2025-10-21 12:20:16,309 - INFO - Fetched page 8 with 50 records
2025-10-21 12:20:16,996 - INFO - Fetched page 9 with 50 records
2025-10-21 12:20:18,777 - INFO - Fetched page 10 with 50 records
2025-10-21 12:20:19,882 - INFO - Fetched page 11 with 50 records
2025-10-21 12:20:21,823 - INFO - Fetched page 12 with 50 records
2025-10-21 12:20:23,370 - INFO - Fetched page 13 with 50 records
2025-10-21 12:20:24,638 - INFO - Fetched page 14 with 50 records
2025-10-21 12:20:25,475 - INFO - Fetched page 15 with 50 records
2025-10-21 12:20:26,340 - INFO - F


Extração concluída! Arquivo salvo em: ./data_lake/bronze/breweries_20251021_122000.json
